In [1]:
# Add imports
import urllib
import networkx as nx
import io
import json
import re
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import random
import pandas as pd
import seaborn as sns
import matplotlib.colors as mcolors
import numpy as np
from itertools import islice
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import matplotlib.pyplot as plt
import numpy as np
import urllib.request
import json
import os
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import re
from wordcloud import *

In [2]:
# Load the network
url = "https://raw.githubusercontent.com/denisababeii/socialgraphs_project/refs/heads/main/celebrities_clean_without_content.graphml"
    
try:
    with urllib.request.urlopen(url) as response:
        data = response.read()
        
    network = nx.read_graphml(io.BytesIO(data))
    print(f"✓ Network loaded successfully!")
    print(f"  Nodes: {network.number_of_nodes()}")
    print(f"  Edges: {network.number_of_edges()}")
        
except Exception as e:
    print(f"✗ Error loading network: {str(e)}")
    network = None

✓ Network loaded successfully!
  Nodes: 11286
  Edges: 125705


In [3]:
era_order_display = [
    'Before Christ (< 30)',
    'Classical Era (30-476)',
    'Middle Ages (476-1450)',
    'Early Modern (1450-1760)',
    'Industrial Era (1760-1840)',
    'Wartime (1840-1945)',
    'Post-War (1945-2025)'
]

def get_era(birth_year):
    """Determine which era a birth year belongs to"""
    if birth_year is None:
        return 'Unknown'
    
    try:
        year = int(birth_year)
    except (ValueError, TypeError):
        return 'Unknown'
    
    if year < 30:
        return 'Before Christ (< 30)'
    elif 30 <= year < 476:
        return 'Classical Era (30-476)'
    elif 476 <= year < 1450:
        return 'Middle Ages (476-1450)'
    elif 1450 <= year < 1760:
        return 'Early Modern (1450-1760)'
    elif 1760 <= year < 1840:
        return 'Industrial Era (1760-1840)'
    elif 1840 <= year < 1945:
        return 'Wartime (1840-1945)'
    elif 1945 <= year <= 2025:
        return 'Post-War (1945-2025)'
    else:
        return 'Unknown'

In [4]:
# Find critical bridge nodes that disconnect large communities
print("="*100)
print("CRITICAL BRIDGE NODES: Single Points of Failure")
print("="*100)

# Get the largest connected component
full_network = network.to_undirected()
largest_component = max(nx.connected_components(full_network), key=len)
largest_subgraph = full_network.subgraph(largest_component).copy()

print(f"\nLargest connected component: {len(largest_component)} nodes")

# Find articulation points (cut vertices)
articulation_points = list(nx.articulation_points(largest_subgraph))
print(f"Found {len(articulation_points)} articulation points (critical bridge nodes)")

# For each articulation point, measure impact of removal
critical_bridges = []

for node in articulation_points:
    # Create network without this node
    temp_network = largest_subgraph.copy()
    temp_network.remove_node(node)
    
    # Find resulting components
    components = list(nx.connected_components(temp_network))
    
    # Get sizes of top 2 components created
    component_sizes = sorted([len(c) for c in components], reverse=True)
    
    if len(component_sizes) >= 2:
        node_name = network.nodes[node].get('Name', 'Unknown')
        occupation = network.nodes[node].get('occupation', 'Unknown')
        era = get_era(network.nodes[node].get('birthyear'))
        is_pope = 'Pope' in network.nodes[node].get('NameURL', '')
        
        critical_bridges.append({
            'node': node,
            'name': node_name,
            'occupation': occupation,
            'era': era,
            'is_pope': is_pope,
            'num_components': len(components),
            'largest_component': component_sizes[0],
            'second_largest': component_sizes[1],
            'total_disconnected': component_sizes[0] + component_sizes[1],
            'impact_score': component_sizes[0] * component_sizes[1]  # Product = measure of "balanced split"
        })

# Sort by impact score (larger = more balanced/severe split)
critical_bridges.sort(key=lambda x: x['impact_score'], reverse=True)

print(f"\nTop 20 Most Critical Bridge Nodes:")
print("-"*130)
print(f"{'Rank':<6} {'Name':<40} {'Occupation':<25} {'Era':<20} {'Pope':<7} {'Split (A/B)':<20}")
print("-"*130)

for rank, bridge in enumerate(critical_bridges[:20], 1):
    pope_mark = "✓" if bridge['is_pope'] else ""
    split_info = f"{bridge['largest_component']}/{bridge['second_largest']}"
    print(f"{rank:<6} {bridge['name'][:38]:<40} {bridge['occupation'][:23]:<25} "
          f"{bridge['era'][:18]:<20} {pope_mark:<7} {split_info:<20}")

# Count how many are Popes
pope_critical = sum(1 for b in critical_bridges if b['is_pope'])
print(f"\n✓ {pope_critical}/{len(critical_bridges)} critical bridges are Popes ({pope_critical/len(critical_bridges)*100:.1f}%)")

# Find the most severe disconnection
if critical_bridges:
    top_bridge = critical_bridges[0]
    print(f"\n" + "="*100)
    print("MOST CRITICAL BRIDGE NODE:")
    print("="*100)
    print(f"Name: {top_bridge['name']}")
    print(f"Occupation: {top_bridge['occupation']}")
    print(f"Era: {top_bridge['era']}")
    print(f"Is Pope: {'Yes' if top_bridge['is_pope'] else 'No'}")
    print(f"\nImpact of removal:")
    print(f"  Creates {top_bridge['num_components']} separate components")
    print(f"  Largest component: {top_bridge['largest_component']} nodes")
    print(f"  Second largest: {top_bridge['second_largest']} nodes")
    print(f"  Total disconnected: {top_bridge['total_disconnected']} nodes")
    print(f"  Impact score: {top_bridge['impact_score']:,}")

CRITICAL BRIDGE NODES: Single Points of Failure

Largest connected component: 11023 nodes
Found 400 articulation points (critical bridge nodes)

Top 20 Most Critical Bridge Nodes:
----------------------------------------------------------------------------------------------------------------------------------
Rank   Name                                     Occupation                Era                  Pope    Split (A/B)         
----------------------------------------------------------------------------------------------------------------------------------
1      Queen Sonja of Norway                    COMPANION                 Wartime (1840-1945           11016/6             
2      Simon Ammann                             SKIER                     Post-War (1945-202           11017/5             
3      Lance Armstrong                          CYCLIST                   Post-War (1945-202           11017/5             
4      Georg Wittig                             CHEMIST       

In [5]:
# Detect communities and analyze critical bridges between them
from networkx.algorithms import community

print("="*100)
print("COMMUNITY DETECTION AND CRITICAL BRIDGES")
print("="*100)

# Detect communities in the largest component
communities = list(community.greedy_modularity_communities(largest_subgraph))
print(f"\nDetected {len(communities)} communities in the largest component")

# Show top 5 communities by size
print("\nTop 5 Communities:")
for i, comm in enumerate(sorted(communities, key=len, reverse=True)[:5], 1):
    print(f"\n  Community {i}: {len(comm)} members")
    
    # Dominant eras
    eras = [get_era(network.nodes[node].get('birthyear')) for node in comm]
    era_counts = Counter(eras)
    print(f"    Top eras: {era_counts.most_common(3)}")
    
    # Dominant occupations
    occupations = [network.nodes[node].get('occupation', 'Unknown') for node in comm]
    occ_counts = Counter(occupations)
    print(f"    Top occupations: {occ_counts.most_common(3)}")

# Assign community labels to nodes
node_to_community = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_to_community[node] = i

# Analyze critical bridges: which communities do they connect?
print("\n" + "="*100)
print("CRITICAL BRIDGES AND THEIR COMMUNITIES")
print("="*100)

for bridge in critical_bridges[:10]:
    node = bridge['node']
    neighbors = list(largest_subgraph.neighbors(node))
    
    # Find which communities the neighbors belong to
    neighbor_communities = set()
    for neighbor in neighbors:
        if neighbor in node_to_community:
            neighbor_communities.add(node_to_community[neighbor])
    
    print(f"\n{bridge['name']}:")
    print(f"  Connects {len(neighbor_communities)} different communities")
    print(f"  Occupation: {bridge['occupation']}")
    print(f"  Era: {bridge['era']}")

COMMUNITY DETECTION AND CRITICAL BRIDGES

Detected 55 communities in the largest component

Top 5 Communities:

  Community 1: 4305 members
    Top eras: [('Wartime (1840-1945)', 1678), ('Early Modern (1450-1760)', 831), ('Industrial Era (1760-1840)', 678)]
    Top occupations: [('POLITICIAN', 1080), ('WRITER', 768), ('RELIGIOUS FIGURE', 332)]

  Community 2: 3343 members
    Top eras: [('Post-War (1945-2025)', 2004), ('Wartime (1840-1945)', 1330), ('Industrial Era (1760-1840)', 5)]
    Top occupations: [('ACTOR', 1120), ('POLITICIAN', 759), ('SINGER', 386)]

  Community 3: 1456 members
    Top eras: [('Post-War (1945-2025)', 1350), ('Wartime (1840-1945)', 104), ('Industrial Era (1760-1840)', 2)]
    Top occupations: [('SOCCER PLAYER', 1028), ('RACECAR DRIVER', 102), ('COACH', 74)]

  Community 4: 670 members
    Top eras: [('Middle Ages (476-1450)', 568), ('Wartime (1840-1945)', 38), ('Early Modern (1450-1760)', 20)]
    Top occupations: [('POLITICIAN', 371), ('RELIGIOUS FIGURE', 163)

In [6]:
# Find nodes that bridge the most communities
print("\n" + "="*100)
print("NODES THAT CONNECT TO THE MOST COMMUNITIES")
print("="*100)

# For each node, count how many different communities its neighbors belong to
node_community_bridges = []

for node in largest_subgraph.nodes():
    neighbors = list(largest_subgraph.neighbors(node))
    
    # Find which communities the neighbors belong to
    neighbor_communities = set()
    for neighbor in neighbors:
        if neighbor in node_to_community:
            neighbor_communities.add(node_to_community[neighbor])
    
    if len(neighbor_communities) > 1:  # Only nodes that bridge at least 2 communities
        node_name = network.nodes[node].get('Name', 'Unknown')
        occupation = network.nodes[node].get('occupation', 'Unknown')
        era = get_era(network.nodes[node].get('birthyear'))
        is_pope = 'Pope' in network.nodes[node].get('NameURL', '')
        degree = len(neighbors)
        
        node_community_bridges.append({
            'node': node,
            'name': node_name,
            'occupation': occupation,
            'era': era,
            'is_pope': is_pope,
            'degree': degree,
            'communities_connected': len(neighbor_communities),
            'bridge_ratio': len(neighbor_communities) / degree if degree > 0 else 0
        })

# Sort by number of communities connected
node_community_bridges.sort(key=lambda x: x['communities_connected'], reverse=True)

print(f"\nTop 20 Nodes by Number of Communities Connected:")
print("-"*130)
print(f"{'Rank':<6} {'Name':<40} {'Occupation':<25} {'Era':<20} {'Pope':<7} {'Communities':<12} {'Degree':<10}")
print("-"*130)

for rank, node in enumerate(node_community_bridges[:20], 1):
    pope_mark = "✓" if node['is_pope'] else ""
    print(f"{rank:<6} {node['name'][:38]:<40} {node['occupation'][:23]:<25} "
          f"{node['era'][:18]:<20} {pope_mark:<7} {node['communities_connected']:<12} {node['degree']:<10}")

# Count how many are Popes in top 20
top_20_popes = sum(1 for node in node_community_bridges[:20] if node['is_pope'])
print(f"\n✓ {top_20_popes}/20 top community bridgers are Popes ({top_20_popes/20*100:.0f}%)")

# Compare with critical bridges
if node_community_bridges:
    print("\n" + "="*100)
    print("COMPARISON: Community Bridges vs Critical Bridges")
    print("="*100)
    
    # Check overlap between top community bridges and critical bridges
    top_community_bridge_nodes = {node['node'] for node in node_community_bridges[:20]}
    critical_bridge_nodes = {bridge['node'] for bridge in critical_bridges[:20]}
    overlap = top_community_bridge_nodes.intersection(critical_bridge_nodes)
    
    print(f"\nNodes in BOTH top 20 lists: {len(overlap)}")
    if overlap:
        print("\nNodes that are both critical bridges AND top community connectors:")
        for node_id in overlap:
            node_name = network.nodes[node_id].get('Name', 'Unknown')
            print(f"  - {node_name}")


NODES THAT CONNECT TO THE MOST COMMUNITIES

Top 20 Nodes by Number of Communities Connected:
----------------------------------------------------------------------------------------------------------------------------------
Rank   Name                                     Occupation                Era                  Pope    Communities  Degree    
----------------------------------------------------------------------------------------------------------------------------------
1      Barack Obama                             POLITICIAN                Post-War (1945-202           11           423       
2      George Bush                              POLITICIAN                Post-War (1945-202           10           310       
3      Pope John Paul II                        RELIGIOUS FIGURE          Wartime (1840-1945   ✓       9            158       
4      Vladimir Putin                           POLITICIAN                Post-War (1945-202           9            151       
5      Ad

In [7]:
# Analyze Queen Sonja's critical bridge role
print("="*100)
print("ANALYZING QUEEN SONJA OF NORWAY'S BRIDGE ROLE")
print("="*100)

# Find Queen Sonja's node
queen_sonja_node = None
for node in network.nodes():
    if 'Queen Sonja of Norway' in network.nodes[node].get('Name', ''):
        queen_sonja_node = node
        break

if queen_sonja_node:
    print(f"\nFound Queen Sonja: {network.nodes[queen_sonja_node].get('Name')}")
    
    # Remove Queen Sonja and find components
    temp_network = largest_subgraph.copy()
    temp_network.remove_node(queen_sonja_node)
    
    components = list(nx.connected_components(temp_network))
    component_sizes = sorted(components, key=len, reverse=True)
    
    print(f"\nComponents after removal:")
    print(f"  Total components: {len(components)}")
    print(f"  Largest component: {len(component_sizes[0])} nodes")
    print(f"  Second largest: {len(component_sizes[1])} nodes")
    
    # Find the smaller disconnected component (the 6 nodes)
    smaller_component = component_sizes[1] if len(component_sizes[1]) <= 10 else component_sizes[-1]
    
    print(f"\n" + "="*50)
    print(f"THE {len(smaller_component)} NODES THAT BECOME DISCONNECTED:")
    print("="*50)
    print(f"\n{'Name':<40} {'Occupation':<30} {'Era':<25}")
    print("-"*95)
    
    for node in smaller_component:
        node_name = network.nodes[node].get('Name', 'Unknown')
        occupation = network.nodes[node].get('occupation', 'Unknown')
        era = get_era(network.nodes[node].get('birthyear'))
        print(f"{node_name:<40} {occupation:<30} {era:<25}")
    
    # Show Queen Sonja's connections to this group
    print(f"\n" + "="*50)
    print("QUEEN SONJA'S CONNECTIONS TO THIS GROUP:")
    print("="*50)
    
    queen_neighbors = set(largest_subgraph.neighbors(queen_sonja_node))
    connections_to_group = queen_neighbors.intersection(smaller_component)
    
    print(f"\nQueen Sonja connects to {len(connections_to_group)} members of this group:")
    for node in connections_to_group:
        node_name = network.nodes[node].get('Name', 'Unknown')
        print(f"  - {node_name}")
    
    # Check if this group has ANY other connections to the main network
    main_component = component_sizes[0]
    alternative_connections = 0
    
    for node in smaller_component:
        neighbors_in_original = set(largest_subgraph.neighbors(node))
        neighbors_in_original.discard(queen_sonja_node)  # Exclude Queen Sonja
        connections_to_main = neighbors_in_original.intersection(main_component)
        alternative_connections += len(connections_to_main)
    
    print(f"\n✓ Alternative connections from this group to main network (excluding Queen Sonja): {alternative_connections}")
    if alternative_connections == 0:
        print("  → Queen Sonja is the ONLY bridge connecting this group to the network!")
else:
    print("\n✗ Queen Sonja of Norway not found in network")

ANALYZING QUEEN SONJA OF NORWAY'S BRIDGE ROLE

Found Queen Sonja: Queen Sonja of Norway

Components after removal:
  Total components: 2
  Largest component: 11016 nodes
  Second largest: 6 nodes

THE 6 NODES THAT BECOME DISCONNECTED:

Name                                     Occupation                     Era                      
-----------------------------------------------------------------------------------------------
Matti Nykänen                            SKIER                          Post-War (1945-2025)     
Thomas Morgenstern                       SKIER                          Post-War (1945-2025)     
Adam Małysz                              SKIER                          Post-War (1945-2025)     
Gregor Schlierenzauer                    SKIER                          Post-War (1945-2025)     
Simon Ammann                             SKIER                          Post-War (1945-2025)     
Janne Ahonen                             SKIER                          Post-War

In [8]:
# Analyze top 5 critical bridges in detail
print("="*100)
print("DETAILED ANALYSIS: TOP 5 CRITICAL BRIDGES")
print("="*100)

for rank, bridge in enumerate(critical_bridges[:5], 1):
    print(f"\n{'='*100}")
    print(f"RANK {rank}: {bridge['name']}")
    print(f"{'='*100}")
    
    node = bridge['node']
    
    print(f"\nBasic Info:")
    print(f"  Occupation: {bridge['occupation']}")
    print(f"  Era: {bridge['era']}")
    print(f"  Is Pope: {'Yes' if bridge['is_pope'] else 'No'}")
    
    # Remove this node and find components
    temp_network = largest_subgraph.copy()
    temp_network.remove_node(node)
    
    components = list(nx.connected_components(temp_network))
    component_sizes = sorted(components, key=len, reverse=True)
    
    print(f"\nImpact of removal:")
    print(f"  Creates {len(components)} separate components")
    print(f"  Largest component: {len(component_sizes[0])} nodes")
    print(f"  Second largest: {len(component_sizes[1])} nodes")
    print(f"  Impact score: {bridge['impact_score']:,}")
    
    # Find the smaller disconnected component
    smaller_component = component_sizes[1]
    
    print(f"\n" + "-"*80)
    print(f"THE {len(smaller_component)} NODES THAT BECOME DISCONNECTED:")
    print("-"*80)
    print(f"\n{'Name':<40} {'Occupation':<30} {'Era':<25}")
    print("-"*95)
    
    for node_id in list(smaller_component)[:20]:  # Show first 20
        node_name = network.nodes[node_id].get('Name', 'Unknown')
        occupation = network.nodes[node_id].get('occupation', 'Unknown')
        era = get_era(network.nodes[node_id].get('birthyear'))
        print(f"{node_name:<40} {occupation:<30} {era:<25}")
    
    if len(smaller_component) > 20:
        print(f"\n... and {len(smaller_component) - 20} more nodes")
    
    # Show connections to this group
    print(f"\n" + "-"*80)
    print(f"{bridge['name']}'S CONNECTIONS TO THIS GROUP:")
    print("-"*80)
    
    bridge_neighbors = set(largest_subgraph.neighbors(node))
    connections_to_group = bridge_neighbors.intersection(smaller_component)
    
    print(f"\n{bridge['name']} connects to {len(connections_to_group)} members of this group:")
    for node_id in list(connections_to_group)[:10]:  # Show first 10
        node_name = network.nodes[node_id].get('Name', 'Unknown')
        print(f"  - {node_name}")
    
    if len(connections_to_group) > 10:
        print(f"  ... and {len(connections_to_group) - 10} more")
    
    # Check if this group has ANY other connections to the main network
    main_component = component_sizes[0]
    alternative_connections = 0
    alt_connection_list = []
    
    for node_id in smaller_component:
        neighbors_in_original = set(largest_subgraph.neighbors(node_id))
        neighbors_in_original.discard(node)  # Exclude the bridge node
        connections_to_main = neighbors_in_original.intersection(main_component)
        alternative_connections += len(connections_to_main)
        if connections_to_main:
            alt_connection_list.extend(connections_to_main)
    
    print(f"\n✓ Alternative connections from this group to main network (excluding {bridge['name']}): {alternative_connections}")
    
    if alternative_connections == 0:
        print(f"  → {bridge['name']} is the ONLY bridge connecting this group to the network!")
    else:
        print(f"\nAlternative bridge nodes ({len(set(alt_connection_list))} unique):")
        for alt_node in list(set(alt_connection_list))[:5]:
            alt_name = network.nodes[alt_node].get('Name', 'Unknown')
            print(f"  - {alt_name}")

print("\n" + "="*100)
print("✓ Top 5 critical bridges analysis complete!")

DETAILED ANALYSIS: TOP 5 CRITICAL BRIDGES

RANK 1: Queen Sonja of Norway

Basic Info:
  Occupation: COMPANION
  Era: Wartime (1840-1945)
  Is Pope: No

Impact of removal:
  Creates 2 separate components
  Largest component: 11016 nodes
  Second largest: 6 nodes
  Impact score: 66,096

--------------------------------------------------------------------------------
THE 6 NODES THAT BECOME DISCONNECTED:
--------------------------------------------------------------------------------

Name                                     Occupation                     Era                      
-----------------------------------------------------------------------------------------------
Matti Nykänen                            SKIER                          Post-War (1945-2025)     
Thomas Morgenstern                       SKIER                          Post-War (1945-2025)     
Adam Małysz                              SKIER                          Post-War (1945-2025)     
Gregor Schlierenzauer    